In [1]:
from pyspark.sql.functions import col, year, month, date_format, monotonically_increasing_id

silver = spark.table("silver_revenue_transactions")

dim_date = (silver.select("order_date").distinct()
    .withColumn("date_key", date_format(col("order_date"), "yyyyMMdd").cast("int"))
    .withColumn("year", year(col("order_date")))
    .withColumn("month_name", date_format(col("order_date"), "MMMM"))
    .withColumn("quarter", ((month(col("order_date"))-1)/3 + 1).cast("int")))
dim_date.write.format("delta").mode("overwrite").saveAsTable("gold_dim_date")

dim_customer = silver.select("customer_id", "customer_name", "region").dropDuplicates(["customer_id"])
dim_customer.write.format("delta").mode("overwrite").saveAsTable("gold_dim_customer")

dim_product = (silver.select("product_name", "product_category").dropDuplicates(["product_name"])
    .withColumn("product_key", monotonically_increasing_id()))
dim_product.write.format("delta").mode("overwrite").saveAsTable("gold_dim_product")

fact = (silver.join(dim_product, on="product_name", how="left")
    .withColumn("date_key", date_format(col("order_date"), "yyyyMMdd").cast("int"))
    .select("order_id", "date_key", "customer_id", "product_key", "channel", "sales_rep",
            "quantity", "unit_price", "discount_pct", "revenue"))
fact.write.format("delta").mode("overwrite").saveAsTable("gold_fact_revenue")

StatementMeta(, a4db59c3-e759-4f90-a018-68b4a5cd489b, 3, Finished, Available, Finished, False)